# Create Structured Passed Bills Dataset

This notebook transforms the raw passed-bill JSON data into a structured Hugging Face dataset with:
- Flat columns extracted from the nested JSON structure
- Official law PDF documents as binary data

Published dataset: https://huggingface.co/datasets/nickbes/lawsofisrael

## Important Notes

**Initiators Field:** The `initiators` field is only populated for private bills (הצעות חוק פרטיות). Government bills (הצעות חוק ממשלתיות) have an empty `initiators` field because they are proposed by the government itself, not individual Knesset members. This is expected behavior from the Knesset API.
- Private bills (`proposal_type == "פרטית"`): Contain Knesset member names in `initiators` (comma-separated string)
- Government bills (`proposal_type == "ממשלתית"`): Empty `initiators` field

Note: Private bill initiators are always Knesset members. Third parties (citizens, organizations, lobbyists) cannot be listed as initiators, though they may draft bills that Knesset members formally propose.


## 1. Setup and Load Raw Data

In [2]:
import json
import time
from pathlib import Path

import requests
from datasets import Dataset, Features, Value
from huggingface_hub import hf_hub_download, notebook_login
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [3]:
# Download raw JSON from Hugging Face
raw_json_path = hf_hub_download(
    repo_id="nickbes/lawsofisrael",
    filename="data/passed_bills_raw.json",
    repo_type="dataset",
)

with open(raw_json_path, "r", encoding="utf-8") as f:
    raw_bills = json.load(f)

print(f"Loaded {len(raw_bills):,} raw bills")
print(f"\nFirst bill structure:")
print(json.dumps(raw_bills[0]["general"], indent=2, ensure_ascii=False)[:500])

Loaded 573 raw bills

First bill structure:
{
  "Id": 1057303,
  "Name": "חוק לתיקון פקודת בתי הסוהר (הארכת הוראות שעה) (תיקוני חקיקה), התשפ\"ו-2026",
  "Continuity": "",
  "Knesset": 25,
  "SubType": "ממשלתית",
  "SubTypeId": null,
  "PrivateNumber": "",
  "Status": "התקבלה בקריאה שלישית",
  "IsInarchive": null,
  "SummaryLaw": "",
  "CommitteeName": "הוועדה לביטחון לאומי",
  "Initiators": "",
  "Joins": "",
  "PublicationSeriesLaw": "ספר החוקים, 29/07/2026, מס' חוברת 3575, עמ' 1048",
  "PublicationSeriesFirstCall": "הצעות חוק הממשלה, 29


## 2. Extract Flat Columns and Filter Bills

In [4]:
def extract_flat_fields(bill: dict) -> dict:
    """Extract flat fields from the nested bill structure."""
    general = bill["general"]
    return {
        "bill_id": general.get("Id"),
        "name": general.get("Name"),
        "status": general.get("Status"),
        "initiators": general.get("Initiators"),
        "knesset_num": general.get("Knesset"),
        "proposal_type": general.get("SubType"),
        "committee_name": general.get("CommitteeName"),
        "commencement_date": general.get("CommencementDate"),
        "summary": general.get("SummaryLaw"),
        "publication_date": general.get("PublicationSeriesLaw"),
        "previous_names": general.get("PreviousNameList"),
    }


def get_official_law_url(bill: dict) -> str | None:
    """Find the official law PDF URL from the legal documents."""
    legal_documents = bill.get("sessionAndDocs", {}).get("LegalDocuments", [])
    official_law = next(
        (doc for doc in legal_documents if doc.get("FileText") == "חוק - פרסום ברשומות"),
        None,
    )
    if official_law and official_law.get("FilePath"):
        return official_law["FilePath"].replace("\\", "/")
    return None

In [5]:
# Process all bills
processed_bills = []
bills_without_pdf = []

for bill in raw_bills:
    flat_fields = extract_flat_fields(bill)
    pdf_url = get_official_law_url(bill)
    
    if pdf_url:
        flat_fields["law_pdf_url"] = pdf_url
        processed_bills.append(flat_fields)
    else:
        bills_without_pdf.append(flat_fields["bill_id"])

print(f"Bills with official PDF: {len(processed_bills):,}")
print(f"Bills without official PDF: {len(bills_without_pdf):,}")
if bills_without_pdf:
    print(f"Bill IDs without PDF: {bills_without_pdf}")

Bills with official PDF: 572
Bills without official PDF: 1
Bill IDs without PDF: [2219672]


In [6]:
# Display sample row
print("Sample processed bill:")
print(json.dumps(processed_bills[0], indent=2, ensure_ascii=False))

Sample processed bill:
{
  "bill_id": 1057303,
  "name": "חוק לתיקון פקודת בתי הסוהר (הארכת הוראות שעה) (תיקוני חקיקה), התשפ\"ו-2026",
  "status": "התקבלה בקריאה שלישית",
  "initiators": "",
  "knesset_num": null,
  "proposal_type": null,
  "committee_name": "הוועדה לביטחון לאומי",
  "commencement_date": "2026-07-29T00:00:00",
  "summary": null,
  "publication_date": null,
  "previous_names": null,
  "law_pdf_url": "https://fs.knesset.gov.il/25/law/25_lsr_14308445.pdf"
}


## 3. Download PDF Documents

In [7]:
# Setup HTTP session with retry logic
HEADERS = {"User-Agent": "lawsofisrael-data-collection/0.1"}
REQUEST_DELAY_SECONDS = 0.2

retry = Retry(
    total=4,
    backoff_factor=0.5,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods={"GET"},
)
http = requests.Session()
http.headers.update(HEADERS)
http.mount("https://", HTTPAdapter(max_retries=retry))

In [8]:
# Test download with first bill
test_url = processed_bills[0]["law_pdf_url"]
print(f"Testing download from: {test_url}")

response = http.get(test_url, timeout=60)
response.raise_for_status()
test_bytes = response.content

print(f"Downloaded {len(test_bytes):,} bytes")
print(f"PDF header: {test_bytes[:8]}")

Testing download from: https://fs.knesset.gov.il/25/law/25_lsr_14308445.pdf
Downloaded 85,279 bytes
PDF header: b'%PDF-1.4'


In [9]:
# Download all PDFs
download_failures = []

for i, bill in enumerate(processed_bills, start=1):
    try:
        response = http.get(bill["law_pdf_url"], timeout=60)
        response.raise_for_status()
        pdf_bytes = response.content
        
        # Verify it's a PDF
        if not pdf_bytes.startswith(b"%PDF"):
            raise ValueError("Downloaded content is not a valid PDF")
        
        bill["law_pdf_bytes"] = pdf_bytes
        
    except (requests.RequestException, ValueError) as e:
        download_failures.append({"bill_id": bill["bill_id"], "error": str(e)})
        bill["law_pdf_bytes"] = None
    
    if i % 50 == 0:
        print(f"Downloaded {i:,} / {len(processed_bills):,} PDFs")
    
    time.sleep(REQUEST_DELAY_SECONDS)

print(f"\nDownload complete!")
print(f"Successes: {sum(1 for b in processed_bills if b['law_pdf_bytes'] is not None):,}")
print(f"Failures: {len(download_failures):,}")

if download_failures:
    print(f"\nFailed downloads:")
    for f in download_failures:
        print(f"  Bill {f['bill_id']}: {f['error']}")

Downloaded 50 / 572 PDFs
Downloaded 100 / 572 PDFs
Downloaded 150 / 572 PDFs
Downloaded 200 / 572 PDFs
Downloaded 250 / 572 PDFs
Downloaded 300 / 572 PDFs
Downloaded 350 / 572 PDFs
Downloaded 400 / 572 PDFs
Downloaded 450 / 572 PDFs
Downloaded 500 / 572 PDFs
Downloaded 550 / 572 PDFs

Download complete!
Successes: 572
Failures: 0


In [10]:
# Remove bills with failed downloads
final_bills = [b for b in processed_bills if b["law_pdf_bytes"] is not None]
print(f"Final dataset size: {len(final_bills):,} bills")

Final dataset size: 572 bills


## 4. Build Dataset with Features Schema

In [11]:
# Define the Features schema
features = Features(
    {
        "bill_id": Value("int64"),
        "name": Value("string"),
        "status": Value("string"),
        "initiators": Value("string"),
        "knesset_num": Value("int64"),
        "proposal_type": Value("string"),
        "committee_name": Value("string"),
        "commencement_date": Value("string"),
        "summary": Value("string"),
        "publication_date": Value("string"),
        "previous_names": Value("string"),
        "law_pdf_url": Value("string"),
        "law_pdf_bytes": Value("large_binary"),
    }
)

print("Features schema:")
for name, feature in features.items():
    print(f"  {name}: {feature}")

Features schema:
  bill_id: Value('int64')
  name: Value('string')
  status: Value('string')
  initiators: Value('string')
  knesset_num: Value('int64')
  proposal_type: Value('string')
  committee_name: Value('string')
  commencement_date: Value('string')
  summary: Value('string')
  publication_date: Value('string')
  previous_names: Value('string')
  law_pdf_url: Value('string')
  law_pdf_bytes: Value('large_binary')


In [12]:
# Create the dataset
dataset = Dataset.from_list(final_bills, features=features)

print(dataset)
print(f"\nDataset features:")
print(dataset.features)

Dataset({
    features: ['bill_id', 'name', 'status', 'initiators', 'knesset_num', 'proposal_type', 'committee_name', 'commencement_date', 'summary', 'publication_date', 'previous_names', 'law_pdf_url', 'law_pdf_bytes'],
    num_rows: 572
})

Dataset features:
{'bill_id': Value('int64'), 'name': Value('string'), 'status': Value('string'), 'initiators': Value('string'), 'knesset_num': Value('int64'), 'proposal_type': Value('string'), 'committee_name': Value('string'), 'commencement_date': Value('string'), 'summary': Value('string'), 'publication_date': Value('string'), 'previous_names': Value('string'), 'law_pdf_url': Value('string'), 'law_pdf_bytes': Value('large_binary')}


In [13]:
# Verify a sample row
sample = dataset[0]
print("Sample row:")
for key, value in sample.items():
    if key == "law_pdf_bytes":
        print(f"  {key}: {len(value):,} bytes" if value else f"  {key}: None")
    else:
        print(f"  {key}: {value}")

Sample row:
  bill_id: 1057303
  name: חוק לתיקון פקודת בתי הסוהר (הארכת הוראות שעה) (תיקוני חקיקה), התשפ"ו-2026
  status: התקבלה בקריאה שלישית
  initiators: 
  knesset_num: None
  proposal_type: None
  committee_name: הוועדה לביטחון לאומי
  commencement_date: 2026-07-29T00:00:00
  summary: None
  publication_date: None
  previous_names: None
  law_pdf_url: https://fs.knesset.gov.il/25/law/25_lsr_14308445.pdf
  law_pdf_bytes: 85,279 bytes


## 5. Push to Hugging Face Hub

In [14]:
# Login to Hugging Face
notebook_login()

In [15]:
# Push to Hub
dataset.push_to_hub(
    repo_id="nickbes/lawsofisrael",
    commit_message="Add structured passed bills dataset with PDF documents",
)

print(f"\nDataset pushed to: https://huggingface.co/datasets/nickbes/lawsofisrael")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Dataset pushed to: https://huggingface.co/datasets/nickbes/lawsofisrael


In [ ]:
# Verify the dataset can be loaded
from datasets import load_dataset

loaded_ds = load_dataset("nickbes/lawsofisrael")
print(loaded_ds)
print(f"\nSample row from loaded dataset:")
sample = loaded_ds["train"][0]
print(f"  bill_id: {sample['bill_id']}")
print(f"  name: {sample['name']}")
print(f"  law_pdf_url: {sample['law_pdf_url']}")
print(f"  law_pdf_bytes: {len(sample['law_pdf_bytes']):,} bytes" if sample['law_pdf_bytes'] else "  law_pdf_bytes: None")